In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [23]:
#this will tell you the missing values in each column
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [8]:
X_train

,age,gender,fever,cough,city
87,47,Male,101.0,Strong,Bangalore
43,22,Female,99.0,Mild,Bangalore
6,14,Male,101.0,Strong,Bangalore
86,25,Male,104.0,Mild,Bangalore
61,81,Female,98.0,Strong,Mumbai
...,...,...,...,...,...
91,38,Male,NaN,Mild,Delhi
46,19,Female,101.0,Mild,Mumbai
79,48,Female,103.0,Mild,Kolkata
90,59,Female,99.0,Strong,Delhi


In [13]:
#Aam Zindagi - column transformation - first we will do impute - fill the missing values
#all the missing values in fever cols got replaced by the mean value of that column
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

#also add the test data
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [14]:
#Ordinal Encoding for 'cough' column
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

#also on the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [15]:
#One Hot Encoding for 'city' and 'gender' column
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

#also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [16]:
#Extracting Age 
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

#also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [17]:
def _ensure_numpy(arr):
# 	# If it's a sparse matrix (or similar), convert to dense numpy array, otherwise ensure numpy array
	if hasattr(arr, "toarray"):
		return arr.toarray()
	return np.asarray(arr)

X_train_transformed = np.concatenate((
	_ensure_numpy(X_train_age),
	_ensure_numpy(X_train_fever),
	_ensure_numpy(X_train_gender_city),
	_ensure_numpy(X_train_cough)
), axis=1)

# also on the test data
X_test_transformed = np.concatenate((
	_ensure_numpy(X_test_age),
	_ensure_numpy(X_test_fever),
	_ensure_numpy(X_test_gender_city),
	_ensure_numpy(X_test_cough)
), axis=1)

X_train_transformed.shape

(80, 7)

In [38]:
from sklearn.compose import ColumnTransformer

In [40]:
#Mentos Zindagi - using ColumnTransformer
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(drop='first', sparse_output=False), ['gender','city'])
    ], remainder='passthrough')

In [41]:
transformer.fit_transform(X_train).shape

(80, 7)

In [43]:
transformer.transform(X_test).shape

(20, 7)